# Phase 1 vs enriched context (Amazon Reviews 2023)

**Phase 1 (baseline)** matches **`colab/llama_sentiment_baseline_train.ipynb`**: **LLaMA-3-8B + LoRA (PEFT)**, same **instruction + CoT + one-shot** prompts and **`extract_rating_from_output`**-style parsing.

**Phase 2 (proposed system)** is **not** implemented end-to-end in this notebook. The full multi-agent design (**LangGraph**, **LanceDB**, **Analyst / Visual Verifier / RAG Prover / Critic**, dissonance, reflection) is documented in **`Agentic_Sentiment_LLaMA3.html`** (§§12–13).

**This notebook only** compares:
- **Phase 1 inference:** review **text only** (baseline prompt).
- **Context-enriched inference:** same trained adapter, but the prompt adds **product metadata** and optionally a **BLIP image caption** as extra text (a lightweight stand-in for multimodal context—not the full Phase 2 stack).

**Dataset:** [McAuley-Lab/Amazon-Reviews-2023](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023) (`trust_remote_code=True`).

**Prerequisite:** Train in `llama_sentiment_baseline_train.ipynb` and upload **`output_final.zip`** (or point `ADAPTER_PATH` to `./output/final`).


## 1. Install dependencies


In [ ]:
%pip install -q datasets transformers accelerate peft safetensors sentencepiece
%pip install -q bitsandbytes
%pip install -q pandas pillow requests torch matplotlib timm


## 2. Configuration


In [ ]:
import os
import random

# --- Adapter from baseline training (Section 7b of llama_sentiment_baseline_train.ipynb) ---
ADAPTER_ZIP = "output_final.zip"  # upload to Colab root, or None
ADAPTER_PATH = os.environ.get("ADAPTER_PATH", "./output/final")
BASE_MODEL = "meta-llama/Meta-Llama-3-8B"

# --- Amazon Reviews 2023 ---
REVIEW_CONFIG = "raw_review_All_Beauty"
META_CONFIG = "raw_meta_All_Beauty"
N_SAMPLES = 12
SEED = 42

# --- Generation (align with baseline eval) ---
MAX_NEW_TOKENS = 128
POOL_SIZE = 5

# --- Optional BLIP caption (CPU-friendly small model) ---
USE_IMAGE_CAPTION = True
BLIP_MODEL = "Salesforce/blip-image-captioning-base"
MAX_IMAGE_BYTES = 2_000_000

random.seed(SEED)


## 3. Load Amazon Reviews 2023 (reviews + metadata)

Merge `raw_review_*` with `raw_meta_*` on `parent_asin`.


In [ ]:
import re
import warnings
from typing import Any, Dict, List, Optional

import pandas as pd
import requests
from datasets import load_dataset
from PIL import Image
from io import BytesIO

warnings.filterwarnings("ignore")

print("Loading reviews:", REVIEW_CONFIG)
rev = load_dataset("McAuley-Lab/Amazon-Reviews-2023", REVIEW_CONFIG, trust_remote_code=True)
rev_split = rev["full"]
n = min(N_SAMPLES, len(rev_split))
if len(rev_split) >= n:
    indices = random.sample(range(len(rev_split)), n)
else:
    indices = list(range(len(rev_split)))
rows = [rev_split[int(i)] for i in indices]

print("Loading metadata:", META_CONFIG)
meta_ds = load_dataset("McAuley-Lab/Amazon-Reviews-2023", META_CONFIG, split="full", trust_remote_code=True)
needed_asins = {
    str(r.get("parent_asin") or r.get("asin") or "") for r in rows
}
needed_asins.discard("")
meta_by_asin = {}
MAX_META_SCAN = 200_000
if needed_asins:
    for i in range(min(len(meta_ds), MAX_META_SCAN)):
        if len(meta_by_asin) >= len(needed_asins):
            break
        r = meta_ds[i]
        pa = str(r.get("parent_asin") or r.get("asin") or "")
        if pa in needed_asins and pa not in meta_by_asin:
            meta_by_asin[pa] = r

def first_image_url(meta: dict) -> Optional[str]:
    im = meta.get("images") or {}
    if isinstance(im, dict):
        for k in ("hi_res", "large", "thumb"):
            lst = im.get(k) or []
            for u in lst:
                if u and isinstance(u, str) and u.startswith("http"):
                    return u
    return None

def safe_details(meta: dict) -> str:
    d = meta.get("details")
    if d is None:
        return ""
    if isinstance(d, str):
        return d[:800]
    return str(d)[:800]

records: List[Dict[str, Any]] = []
for row in rows:
    pa = str(row.get("parent_asin") or row.get("asin") or "")
    meta = meta_by_asin.get(pa, {})
    title = (row.get("title") or meta.get("title") or "").strip()
    text = (row.get("text") or "").strip()
    rating = row.get("rating")
    try:
        gt = int(round(float(rating)))
    except (TypeError, ValueError):
        gt = 3
    gt = max(1, min(5, gt))
    cat = (meta.get("main_category") or REVIEW_CONFIG.replace("raw_review_", "").replace("_", " ") or "Unknown")
    img_url = first_image_url(meta)
    records.append(
        {
            "parent_asin": pa,
            "review_title": title,
            "review_text": text[:4000],
            "ground_truth_stars": gt,
            "main_category": cat,
            "meta_title": (meta.get("title") or title)[:500],
            "details": safe_details(meta),
            "image_url": img_url,
        }
    )

df_raw = pd.DataFrame(records)
print(df_raw[["ground_truth_stars", "main_category"]].head())
print("Rows:", len(df_raw))


## 4. Optional: image → caption (BLIP)

Runs **before** loading LLaMA to reduce peak VRAM. Caption is **text only** in the enriched prompt—not the Phase 2 Visual Verifier from the HTML doc.


In [ ]:
caption_by_idx: Dict[int, str] = {}

if USE_IMAGE_CAPTION:
    import torch
    from transformers import BlipProcessor, BlipForConditionalGeneration

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("BLIP device:", device)
    proc = BlipProcessor.from_pretrained(BLIP_MODEL)
    blip = BlipForConditionalGeneration.from_pretrained(BLIP_MODEL).to(device)
    blip.eval()

    def caption_from_url(url: str) -> str:
        try:
            r = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
            r.raise_for_status()
            if len(r.content) > MAX_IMAGE_BYTES:
                return ""
            img = Image.open(BytesIO(r.content)).convert("RGB")
            inputs = proc(images=img, return_tensors="pt").to(device)
            out = blip.generate(**inputs, max_new_tokens=40)
            return proc.tokenizer.decode(out[0], skip_special_tokens=True).strip()
        except Exception as e:
            return f"[image unavailable: {e}]"

    for i, row in df_raw.iterrows():
        u = row.get("image_url")
        if isinstance(u, str) and u.startswith("http"):
            caption_by_idx[i] = caption_from_url(u)
        else:
            caption_by_idx[i] = ""
else:
    caption_by_idx = {i: "" for i in range(len(df_raw))}

df_raw["image_caption"] = [caption_by_idx.get(i, "") for i in range(len(df_raw))]
print(df_raw[["image_caption"]].head(3))


## 5. Prompt helpers (same as baseline notebook)


In [ ]:
from dataclasses import dataclass
from typing import Optional

SENTIMENT_INSTRUCTION = (
    "Evaluate the sentiment expressed in user reviews and classify each one according to its sentiment rating. "
    "Use a five-point scale: 1-2 negative, 3 neutral, 4-5 positive."
)
RATING_DESCRIPTIONS = {
    1: "Comments show a high level of dissatisfaction and negativity (rating 1).",
    2: "Comments show dissatisfaction (rating 2).",
    3: "Comments are mixed or neutral (rating 3).",
    4: "Comments show satisfaction (rating 4).",
    5: "Comments show strong satisfaction and positivity (rating 5).",
}


@dataclass
class DataConfig:
    use_cot: bool = True
    cot_phrase: str = "Let's take it one step at a time."
    use_one_shot: bool = True


data_cfg = DataConfig()


def format_one_shot(review, rating, cfg: DataConfig):
    desc = RATING_DESCRIPTIONS.get(rating, f"Rating {rating}.")
    return f"Review: {review}\nSentiment (1-5): {rating}. {desc}"


def build_prompt(review: str, cfg: DataConfig, one_shot_example: Optional[str] = None) -> str:
    parts = [SENTIMENT_INSTRUCTION]
    if cfg.use_cot:
        parts.append(cfg.cot_phrase)
    if one_shot_example and cfg.use_one_shot:
        parts.append("\n\nExample:\n" + one_shot_example)
    parts.append("\n\nReview to classify:\n" + review)
    parts.append(
        "\nAfter your reasoning, end with exactly one line starting with \"Sentiment (1-5):\" "
        "followed by the rating 1–5 and a short justification (paper-style output)."
    )
    parts.append("\nSentiment (1-5):")
    return "\n".join(parts)


def build_prompt_with_product_context(
    review: str,
    cfg: DataConfig,
    one_shot_example: Optional[str],
    meta_title: str,
    category: str,
    details: str,
    image_caption: str,
) -> str:
    """Same as baseline prompt, plus a block of product context (metadata + optional image caption)."""
    ctx_lines = [
        "Product context (for context-enriched inference demo — not the full Phase 2 agentic system):",
        f"- Category: {category}",
        f"- Product title: {meta_title}",
        f"- Details: {details[:600]}",
    ]
    cap = (image_caption or "").strip()
    if cap:
        ctx_lines.append(f"- Image description (BLIP): {cap[:500]}")
    ctx = "\n".join(ctx_lines)
    parts = [SENTIMENT_INSTRUCTION, ctx]
    if cfg.use_cot:
        parts.append(cfg.cot_phrase)
    if one_shot_example and cfg.use_one_shot:
        parts.append("\n\nExample:\n" + one_shot_example)
    parts.append("\n\nReview to classify:\n" + review)
    parts.append(
        "\nAfter your reasoning, end with exactly one line starting with \"Sentiment (1-5):\" "
        "followed by the rating 1–5 and a short justification (paper-style output)."
    )
    parts.append("\nSentiment (1-5):")
    return "\n".join(parts)


def create_one_shot_pool(samples, cfg: DataConfig, pool_size: int = 5):
    by_rating = {r: [] for r in range(1, 6)}
    for s in samples:
        r = s.get("ground_truth_stars") or s.get("rating")
        if r in by_rating:
            by_rating[r].append(s)
    pool = []
    for r in range(1, 6):
        lst = by_rating[r]
        if lst:
            pool.append(random.choice(lst))
    rng = random.Random(SEED)
    extra = [s for s in samples if s not in pool]
    rng.shuffle(extra)
    while len(pool) < pool_size and extra:
        pool.append(extra.pop())
    return pool[:pool_size]


def extract_rating_from_output(text: str) -> int:
    """Parse rating; prefer the *last* 'Sentiment (1-5):' line (CoT may mention other digits)."""
    text = (text or "").strip()
    if not text:
        return 3
    low = text.lower()
    key = "sentiment (1-5)"
    if key in low:
        idx = low.rfind(key)
        tail = text[idx : idx + 500]
        m = re.search(r"[Ss]entiment\s*\(1-5\)\s*[:=]\s*([1-5])", tail)
        if m:
            return int(m.group(1))
        m = re.search(r"[:=]\s*([1-5])\b", tail)
        if m:
            return int(m.group(1))
    m = re.search(r"[Rr]ating\s*[:=]\s*([1-5])\b", text)
    if m:
        return int(m.group(1))
    for line in reversed(text.splitlines()):
        line = line.strip()
        m = re.match(r"^([1-5])\s*[\.\:)]", line)
        if m:
            return int(m.group(1))
        m = re.match(r"^([1-5])$", line)
        if m:
            return int(m.group(1))
    m2 = re.search(r"\b([1-5])\b", text)
    if m2:
        return int(m2.group(1))
    digits = re.findall(r"[1-5]", text)
    if digits:
        return int(digits[-1])
    return 3


## 6. Load LLaMA-3-8B + PEFT adapter (baseline Section 7b)


In [ ]:
import zipfile

if ADAPTER_ZIP and os.path.isfile(ADAPTER_ZIP):
    print("Unzipping", ADAPTER_ZIP, "...")
    os.makedirs("output", exist_ok=True)
    with zipfile.ZipFile(ADAPTER_ZIP) as z:
        z.extractall(".")
    print("Done.")

if not os.path.isdir(ADAPTER_PATH):
    raise FileNotFoundError(
        f"Adapter not found at {ADAPTER_PATH}. Train with llama_sentiment_baseline_train.ipynb, "
        f"zip output as output_final.zip, upload, or set ADAPTER_PATH."
    )

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

try:
    import bitsandbytes  # noqa: F401
    _use_4bit = True
except Exception:
    _use_4bit = False

print("Loading tokenizer from", ADAPTER_PATH)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {"trust_remote_code": True, "device_map": "auto", "torch_dtype": torch.float16}
if _use_4bit:
    from transformers import BitsAndBytesConfig

    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )
print("Loading base", BASE_MODEL, "(4-bit)" if _use_4bit else "(fp16)")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()
print("Model ready.")


## 7. Inference: Phase 1 (text-only) vs context-enriched (same adapter)


In [ ]:
sample_dicts = df_raw.to_dict("records")
one_shot_pool = create_one_shot_pool(sample_dicts, data_cfg, pool_size=POOL_SIZE)


def pick_one_shot_for_row(row, pool):
    gt = int(row["ground_truth_stars"])
    candidates = [p for p in pool if int(p["ground_truth_stars"]) != gt]
    if not candidates:
        candidates = pool
    return format_one_shot(
        candidates[0]["review_text"],
        int(candidates[0]["ground_truth_stars"]),
        data_cfg,
    )


@torch.inference_mode()
def generate_rating(prompt: str) -> int:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    gen_cfg = GenerationConfig(
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    out = model.generate(**inputs, generation_config=gen_cfg)
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
    return extract_rating_from_output(gen)


def predict_phase1(row) -> int:
    one = pick_one_shot_for_row(row, one_shot_pool)
    prompt = build_prompt(row["review_text"], data_cfg, one_shot_example=one)
    return generate_rating(prompt)


def predict_context_enriched(row) -> int:
    one = pick_one_shot_for_row(row, one_shot_pool)
    prompt = build_prompt_with_product_context(
        row["review_text"],
        data_cfg,
        one,
        str(row.get("meta_title") or ""),
        str(row.get("main_category") or ""),
        str(row.get("details") or ""),
        str(row.get("image_caption") or ""),
    )
    return generate_rating(prompt)


## 8. Run comparison & plot


In [ ]:
results = []
for _, row in df_raw.iterrows():
    r = row.to_dict()
    gt = int(r["ground_truth_stars"])
    p1 = predict_phase1(r)
    p2 = predict_context_enriched(r)
    results.append({
        "gt": gt,
        "phase1": p1,
        "enriched": p2,
        "snippet": (r["review_text"] or "")[:80].replace("\n", " "),
    })

df_res = pd.DataFrame(results)
print(df_res.to_string(index=False))

def acc(series, gt):
    return (series == gt).mean() if len(gt) else 0.0

print("Accuracy Phase 1 (text-only):", acc(df_res["phase1"], df_res["gt"]))
print("Accuracy context-enriched:", acc(df_res["enriched"], df_res["gt"]))


In [ ]:
import matplotlib.pyplot as plt

labels = [f"s{i}" for i in range(len(df_res))]
x = range(len(df_res))
plt.figure(figsize=(10, 4))
plt.plot(x, df_res["gt"], "ko-", label="Ground truth")
plt.plot(x, df_res["phase1"], "s-", label="Phase 1 (text-only)")
plt.plot(x, df_res["enriched"], "^-", label="Context-enriched (same adapter)")
plt.xticks(x, labels)
plt.ylabel("Star rating (1–5)")
plt.xlabel("Sample")
plt.legend()
plt.title("Baseline LLaMA+LoRA vs metadata+image caption in prompt")
plt.tight_layout()
plt.show()


## 9. How this maps to your documents

| Item | Where |
|------|--------|
| **Phase 1** training, prompts, eval | `colab/llama_sentiment_baseline_train.ipynb` |
| **Phase 2** architecture (LangGraph, LanceDB, agents, dissonance) | `Agentic_Sentiment_LLaMA3.html` §§12–13 |
| **This notebook** | Same adapter as Phase 1; extra context in the prompt only — **not** the full Phase 2 pipeline. |

For your presentation, show the HTML architecture for Phase 2 and this notebook (or eval metrics from the baseline notebook) for Phase 1.
